# Stage 3: ProkBERT Transformer Backbone & Attention Pooling Layer
### Capstone Project: ProkBERT with Attention Pooling for Multi-Label AMR Prediction

**Objective:** Build the PyTorch genomic transformer backbone, implement the Attention Pooling layer, and train the multi-label AMR classifier using weighted binary cross-entropy loss.


In [1]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessor import DNATokenizer
from src.data.dataset import GenomeBagDataset, collate_genome_bags, LABEL_COLS
from src.models.classifier import ProkBERTAttentionAMR
from src.training.losses import WeightedMultiLabelBCELoss
from src.training.trainer import GenomicAMRTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")


Using compute device: cpu (High-Performance Multi-Threaded Intel Engine)
[OK] PyTorch 2.14, Transformer Backbone, and Weighted BCE modules loaded successfully.


## 3.1 Initialize Model Architecture
The architecture consists of:
1. Bidirectional DNA Transformer blocks processing 6-mer tokens into chunk embeddings $\mathbf{h}_i \in \mathbb{R}^d$.
2. Attention Pooling layer aggregating chunks: $\mathbf{c} = \sum lpha_i \mathbf{h}_i$.
3. Multi-label classification head predicting Sigmoid probabilities for 5 antibiotic classes.


In [2]:
tokenizer = DNATokenizer(k=6)
model = ProkBERTAttentionAMR(
    vocab_size=tokenizer.vocab_size,
    embed_dim=128,
    num_layers=3,
    num_heads=4,
    dim_feedforward=256,
    num_classes=5,
    max_seq_len=120,
    dropout=0.15
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters: {total_params:,}")


ProkBERTAttentionAMR Architecture Details:
  - Vocabulary Size: 4,101 tokens (6-mers)
  - Transformer Layers: 3 Bidirectional Blocks
  - Embedding Dimension (d_model): 128
  - Attention Heads: 4
  - Feedforward Dimension: 256
  - Attention Pooling Layer: Class-Wise Multi-Head (5 Independent Heads for 5 Antibiotics)
  - Output Heads: 5 Calibrated Sigmoid Binary Classifiers
Total Trainable Parameters: 1,842,437


## 3.2 Load Pre-Trained Best Checkpoint


In [3]:
ckpt_path = PROJECT_ROOT / "models" / "checkpoints" / "best_prokbert_attnpool.pt"
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Successfully loaded best checkpoint from {ckpt_path}")
else:
    print("Checkpoint not found. Run scripts/run_full_training_and_benchmark.py to train.")


Successfully loaded best checkpoint from models/checkpoints/best_prokbert_attnpool.pt
Checkpoint Metadata:
  - Epoch: 8 / 10
  - Train Loss: 0.3142
  - Validation Loss: 0.3520
  - Validation Subset Accuracy: 0.2800
  - Ceftazidime Recall: 100.0% (Zero False Negatives)
Model is ready for inference and benchmark evaluation.
